# Word Embeddings: Domain-Specific vs. Pretrained Models
**Cameron Batts**

Python · NLP · Gensim · Word2Vec · FastText · CBOW · Skip-gram

## Project overview
Explore how corpus domain, training architecture, and subword information affect the nearest neighbors returned by word embeddings. Train four 100-dimensional models on 2,000 movie reviews and compare selected queries with 300-dimensional pretrained Google News embeddings.

## Business relevance
Embedding similarity can support content discovery, search, and vocabulary exploration. This project examines whether nearby terms reflect useful meaning or merely shared spelling and corpus context; it does not measure production search performance.

## Data and attribution
The movie-review corpus is distributed by NLTK and associated with Pang and Lee (2004), *Sentimental Education: Sentiment Analysis Using Subjectivity Summarization Based on Minimum Cuts*. Source: https://www.cs.cornell.edu/people/pabo/movie-review-data/ ; paper: https://aclanthology.org/P04-1035/ . Attribution is retained. Raw review excerpts and saved person-name neighbors are omitted from this presentation.

## Execution status
Selected aggregate and neighbor results are retained from the supplied notebook. The original training emitted repeated Gensim numerical warnings (`our_dot_float`); models and scores have not been independently retrained or verified. The cleaned code is syntax-checked, not execution-validated. Original four-worker training is retained; results may vary despite the fixed seed.

In [ ]:
import pandas as pd
import numpy as np
import gensim
import nltk
from nltk.corpus import movie_reviews
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec, FastText
import gensim.parsing.preprocessing as preprocess

nltk.download(['movie_reviews', 'wordnet', 'averaged_perceptron_tagger_eng'], quiet=True)
doc = [' '.join(movie_reviews.words(file_id)) for file_id in movie_reviews.fileids()]
text_series = pd.Series(doc)
text_series.str.len().describe()


count     2000.0000
mean      3904.2600
std       1718.6621
min         91.0000
25%       2745.5000
50%       3640.5000
75%       4730.5000
max      15097.0000
dtype: float64

## Text preprocessing
Remove punctuation, numbers, stopwords, HTML tags, and repeated whitespace; apply part-of-speech-aware WordNet lemmatization and finish with lowercase tokenization. The final Gensim tokenization also filters tokens by its default length rules. Proper names can remain in the underlying corpus and may appear when the notebook is rerun.

In [ ]:
CUSTOM_FILTERS = [preprocess.strip_punctuation,
    preprocess.strip_numeric,
    preprocess.remove_stopwords, 
    preprocess.strip_tags, 
    preprocess.strip_multiple_whitespaces,]

In [ ]:
def lemmatize_text(token_list, wnl):
  """
  This function tags the pos of the input tokens,
  and lemmatize the tokens.
  """
  # POS tag each word
  for word, tag in nltk.pos_tag(token_list):
    # Mapping the pos tags to the types supported by wnl
    if tag.startswith("NN"):
      yield wnl.lemmatize(word, pos='n')
    elif tag.startswith('VB'):
      yield wnl.lemmatize(word, pos='v')
    elif tag.startswith('JJ'):
      yield wnl.lemmatize(word, pos='a')
    elif tag.startswith('RB'):
      yield wnl.lemmatize(word, pos='r')
    else:
      yield wnl.lemmatize(word)

# Remove punctuations, digits, and stopwords with your defined CUSTOM_FILTERS
processed_text = text_series.apply(lambda x: preprocess.preprocess_string(x, CUSTOM_FILTERS))

# Lemmatize the tokens
wnl = nltk.WordNetLemmatizer()
processed_text = processed_text.apply(lambda x: " ".join(lemmatize_text(x, wnl)))

# Lowercase the letters and return a token list of each review
processed_text = processed_text.apply(lambda x: gensim.utils.simple_preprocess(x, deacc=False))

## Four model configurations
Each model uses vector_size=100, window=5, min_count=5, epochs=10, seed=42, and workers=4. Word2Vec learns whole-word vectors; FastText also uses character subword representations. CBOW predicts a target from context, while skip-gram predicts context from a target. The same review corpus supports all four configurations.

In [ ]:
from gensim.models import Word2Vec, FastText
# vector_size = number of dimensions in each word vector
# window = how many words on each side count as context
# min_count = ignore words that show up fewer than 5 times
# sg = 0 means CBOW, sg = 1 means skip-gram
w2v_cbow = Word2Vec(sentences=processed_text, vector_size=100, window=5, min_count=5, sg=0, workers=4, epochs=10, seed=42)
w2v_sg = Word2Vec(sentences=processed_text, vector_size=100, window=5, min_count=5, sg=1, workers=4, epochs=10, seed=42)
ft_cbow = FastText(sentences=processed_text, vector_size=100, window=5, min_count=5, sg=0, workers=4, epochs=10, seed=42)
ft_sg = FastText(sentences=processed_text, vector_size=100, window=5, min_count=5, sg=1, workers=4, epochs=10, seed=42)


## Inspect nearest neighbors
Cosine similarity identifies embedding neighbors, which can include related topics, morphological variants, and unrelated-looking corpus associations. It does not establish synonymy or downstream accuracy. The examples below retain selected non-person neighbors from the original output; they are not complete top-ten lists.

### Word2Vec skip-gram — disney

In [ ]:
# top-10 most similar words for "disney" using w2v_sg
w2v_sg.wv.most_similar(positive="disney")

animate: 0.7660
animated: 0.7395
animation: 0.7316
mermaid: 0.7021
hunchback: 0.6921


### FastText skip-gram — oscar

In [ ]:
# top-10 most similar words for "oscar" using ft_sg
ft_sg.wv.most_similar(positive="oscar")

nominee: 0.8377
nomi: 0.8174
nominate: 0.8037
academy: 0.7960
nominated: 0.7956
nomination: 0.7895
award: 0.7798
winner: 0.7198
globe: 0.7043


### FastText CBOW — actress

In [ ]:
# top-10 most similar words for "actress" using ft_cbow
ft_cbow.wv.most_similar(positive="actress")

actresses: 0.9722
roles: 0.8735
role: 0.8651
support: 0.8449
actor: 0.8403
charisma: 0.8335
performer: 0.8331


### Word2Vec CBOW — kung + fu

In [ ]:
# top-10 most similar words for "kung" and "fu" using w2v_cbow
w2v_cbow.wv.most_similar(positive=["kung", "fu"])

choreograph: 0.8792
heavy: 0.8194
choreography: 0.8118
spectacular: 0.8095
slo: 0.8040
stunt: 0.7942


### Word2Vec skip-gram — vampire

In [ ]:
#Top-10 most similar words for 'vampire' using w2v_sg
w2v_sg.wv.most_similar(positive="vampire")

slayer: 0.6586
vampires: 0.6180
sunlight: 0.5917
lore: 0.5854
bitten: 0.5839
myth: 0.5704


### FastText skip-gram — vampire

In [ ]:
#Top-10 most similar words for 'vampire' using ft_sg
ft_sg.wv.most_similar(positive="vampire")

vampires: 0.9571
vamp: 0.8254
empire: 0.7256
superheroes: 0.5866


### Word2Vec CBOW — vampire

In [ ]:
#Top-10 most similar words for 'vampire' using w2v_cbow
w2v_cbow.wv.most_similar(positive="vampire")

mar: 0.7687
friday: 0.7512
runner: 0.7509
soldier: 0.7381
battlefield: 0.7210
invasion: 0.7192
disintegrate: 0.7184
warrior: 0.7174
planet: 0.7148


### FastText CBOW — vampire

In [ ]:
#Top-10 most similar words for 'vampire' using ft_cbow
ft_cbow.wv.most_similar(positive="vampire")

vampires: 0.9445
vamp: 0.9130
empire: 0.8630
urban: 0.8214
trooper: 0.8204
error: 0.8111
mirror: 0.8088
eraser: 0.8049
laser: 0.8015
cream: 0.7790


## Domain and subword observations
For the animation-studio query, Word2Vec skip-gram returned animation-related terms such as animate (0.7660) and animation (0.7316). For the award query, FastText skip-gram returned nominee (0.8377) and nomination (0.7895).

For vampire, FastText skip-gram returned vampires (0.9571) but also empire (0.7256); FastText CBOW likewise included empire, mirror, and laser. Shared character fragments can influence similarity without guaranteeing a strong conceptual relationship. Word2Vec skip-gram returned genre-related words such as slayer, sunlight, lore, and bitten. These observations are qualitative, not a model ranking.

## Optional pretrained benchmark
Google News vectors require a substantial download and memory allocation. Enable the following cell only when resources are available. It loads pretrained weights at runtime rather than bundling them with the project. The pretrained model has different dimensionality, vocabulary and training data, so raw similarity scores are not a common quality scale.

In [ ]:
RUN_PRETRAINED = False
if RUN_PRETRAINED:
    import gensim.downloader as api
    pretrained = api.load('word2vec-google-news-300')
    for token in ['computer', 'vampire']:
        print(token, pretrained.most_similar(positive=token, topn=10))


## General vocabulary comparison
The original Google News output for computer included computers (0.7979), laptop (0.6640), and laptop_computer (0.6549). The domain-trained FastText CBOW output included interpreter (0.8746), composer (0.8580), and consumer (0.8564). The larger scores in the latter do not make these stronger general-computing matches.

For vampire, the pretrained model included vampires (0.8276), werewolf (0.7312), and undead (0.6609). The domain-trained FastText skip-gram model mixed morphology with film-specific associations. The two corpora encode different contexts.

## Applications and limitations
- Domain embeddings can expose useful specialized vocabulary, but small-corpus associations may not generalize.
- FastText can synthesize vectors for unseen words using subword components; a synthesized vector is not evidence of reliable meaning. Word2Vec requires a learned vocabulary entry.
- Mean word vectors provide a simple sentence representation but lose word order. Sentence-level representations should be evaluated for the intended task.
- The 2,000-review corpus reflects a limited, historical movie domain. No relevance judgments, downstream benchmark, or quantitative model-quality comparison was completed.
- Address the original numerical warnings, record dependency versions, repeat training, and use a labeled retrieval benchmark before selecting a model for deployment.

## Author
Cameron Batts